In [113]:
#1 필요한 모듈 임포트
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import tkinter as tk
from tkinter import filedialog
import requests
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import re
headers={'User-Agent':'Mozilla/5.0 (<system-information>) <platform> (<platform-details>) <extensions>'}
options = Options()
options.add_argument("--headless=new")          # 창 안 띄우는 모드
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

In [ ]:

# 데이터 파일 업로드 및 기본 전처리
root=tk.Tk()
root.withdraw()
file_path = filedialog.askopenfilename(
    title="파일 선택",
    filetypes=[("모든 파일", "*.*"), ("엑셀 파일", "*.xlsx *.xls")]
)
if file_path:
    df_raw=pd.read_excel(
        file_path,
        header=7,
        sheet_name='result'
        )
    df_raw.drop(['조회수','좋아요수','RT수'], axis=1, inplace=True)

if not file_path:
    print('파일 선택이 취소되었습니다.')



In [115]:
def react_extractor(url):
    if 'news.naver.com' in url:
        try:
            driver.get(url)

            WebDriverWait(driver, 10).until(
                lambda d: re.search(r'\d+', d.find_element(By.ID, "comment_count").get_attribute("textContent") or "")
            )

            elem = driver.find_element(By.ID, "comment_count")
            text = elem.get_attribute("textContent").strip()

            m = re.search(r'\d+', text)
            return int(m.group()) if m else None

        except (TimeoutException, NoSuchElementException):
            return None
    else:
        return None

In [116]:
df_raw['updated_react']=df_raw['URL'].apply(lambda x: react_extractor(x))
df_raw.dropna(axis=0, inplace=True)
df_raw['diff']=df_raw['updated_react']-df_raw['댓글수'].astype(int)
df_raw.to_excel('result.xlsx', header=True)


In [ ]:
dd